In [1]:
from bs4 import BeautifulSoup
import os
import csv

month_dict = { 'Jan' : '01', 'Feb' : '02', 'Mar' : '03', 'Apr' : '04',
             'May' : '05', 'Jun' : '06', 'Jul' : '07', 'Aug' : '08',
             'Sept' : '09', 'Oct' : '10', 'Nov' : '11', 'Dec' : '12' }

def extract_history(file_path, etf_name):
    with open(os.path.join(file_path, etf_name + '.txt'), 'r') as txt_file:
        history = txt_file.read()

    soup = BeautifulSoup(history, features='html.parser')
    tbody = soup.table.tbody

    price_dates = []
    price_history = []
    adj_price_dates = []
    adj_price_history = []
    divend_dates = []
    divend_history = []

    for tr in tbody.contents:
        tds = tr.contents

        # 날짜 체크
        date_origin = tds[0].contents[0]
        dd, mm, yyyy= date_origin.split()
        
        date_format = '{0:0>4}'.format(yyyy) + '-' + '{0:0>2}'.format(month_dict[mm]) + '-' + '{0:0>2}'.format(dd)

        if len(tds) == 4:
            divend = tds[2].span.contents[0]
            if ":" in divend:
                continue
            divend_dates.append(date_format)
            divend_history.append(divend)
        else:
            price = tds[4].contents[0]
            price_dates.append(date_format)
            price_history.append(price)
            adj_price = tds[5].contents[0]
            adj_price_dates.append(date_format)
            adj_price_history.append(adj_price)

    with open(os.path.join(file_path, etf_name + '.csv'), 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(reversed(price_dates))
        writer.writerow(reversed(price_history))
        writer.writerow(reversed(adj_price_dates))
        writer.writerow(reversed(adj_price_history))
        writer.writerow(reversed(divend_dates))
        writer.writerow(reversed(divend_history))

In [2]:
file_path = './ETF'
extract_history(file_path, 'qqq')
extract_history(file_path, 'schd')
extract_history(file_path, 'dgrw')
extract_history(file_path, 'spy')